<a href="https://colab.research.google.com/github/NikethnaSri-AI/ai-learning-notebooks/blob/main/reward_modeling_prac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q torch==2.3.1 datasets==3.2.0 trl==0.11 \
transformers==4.43.4 peft==0.14.0 huggingface_hub==0.36.0 \
nltk==3.9.1 rouge_score==0.1.2 bitsandbytes==0.43.1 \
matplotlib==3.10.0 rich==13.7.1

In [ ]:
import json
import warnings
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset, DatasetDict
from transformers import (GPT2Tokenizer, GPT2ForSequenceClassification, TrainingArguments,)
from peft import LoraConfig, TaskType
from trl import RewardTrainer

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
#Helper functions
def save_to_json(data, file_path):
    """Save a dictionary to a JSON file."""
    with open(file_path, "w") as json_file:
        json.dump(data, json_file, indent=4)

    print(f"Data successfully saved to {file_path}")


def load_from_json(file_path):
    """Load data from a JSON file."""
    with open(file_path, "r") as json_file:
        return json.load(json_file)


### Dataset

In [ ]:
dataset = load_dataset("Dahoas/synthetic-instruct-gptj-pairwise")
print(dataset)

In [ ]:
for i in range(5):
    print("Prompt:")
    print(dataset["train"][i]["prompt"], "\n")

    print("Chosen:")
    print(dataset["train"][i]["chosen"], "\n")

    print("Rejected:")
    print(dataset["train"][i]["rejected"], "\n")
    print("-" * 60)

In [ ]:
model_name_or_path = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name_or_path, use_fast=True)
model = GPT2ForSequenceClassification.from_pretrained(model_name_or_path, num_labels=1)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
max_length = 1024

In [ ]:
get_res=lambda dataset,res:[  "\n\nHuman: "+prompt + "\n\nAssistant: "+resp for prompt, resp in zip(dataset["train"]["prompt"], dataset["train"][res])]

In [ ]:
chosen_samples=get_res( dataset,'chosen')
rejected_samples=get_res( dataset,'rejected')
print('chosen',chosen_samples[0])
print('rejected',rejected_samples[0])

In [ ]:
def add_combined_columns(example):
    example["prompt_chosen"] = (
        "\n\nHuman: " + example["prompt"]
        + "\n\nAssistant: " + example["chosen"]
    )

    example["prompt_rejected"] = (
        "\n\nHuman: " + example["prompt"]
        + "\n\nAssistant: " + example["rejected"]
    )

    return example

In [ ]:
dataset["train"] = dataset["train"].map(add_combined_columns)

In [ ]:
print(dataset["train"][0]["prompt_chosen"])
print("-" * 60)
print(dataset["train"][0]["prompt_rejected"])

In [ ]:
get_max_len= lambda samples: max([len(sample) for sample in samples])
get_max_len

In [ ]:
print("rejected samples length",get_max_len(rejected_samples))
print("chosen samples length",get_max_len(chosen_samples))

In [ ]:
find_short = lambda dataset, max_length: [
    i for i, (chosen, rejected) in enumerate(zip(dataset['prompt_chosen'], dataset['prompt_rejected']))
    if len(chosen) < max_length or len(rejected) < max_length
]

In [ ]:
max_length=1024
subset_indices=find_short (dataset['train'], max_length)
dataset['train'] = dataset['train'].select(subset_indices)
subset_indices[0:10]

In [ ]:
def preprocess_function(examples):
    tokenized_chosen = tokenizer(
        examples["prompt_chosen"],
        truncation=True,
        max_length=max_length,
        padding="max_length"
    )

    tokenized_rejected = tokenizer(
        examples["prompt_rejected"],
        truncation=True,
        max_length=max_length,
        padding="max_length"
    )

    return {
        "input_ids_chosen": tokenized_chosen["input_ids"],
        "attention_mask_chosen": tokenized_chosen["attention_mask"],
        "input_ids_rejected": tokenized_rejected["input_ids"],
        "attention_mask_rejected": tokenized_rejected["attention_mask"],
    }

In [ ]:
example=preprocess_function(dataset['train'][0])
example.keys()

In [ ]:
train_str={'chosen': [sample for sample in dataset['train'] ['prompt_chosen']], 'rejected':[sample for sample in dataset['train'] ['prompt_rejected']]}

In [ ]:
dataset['train'] = dataset['train'].map(preprocess_function, batched=True, remove_columns=['prompt',"chosen", "rejected",'prompt_chosen', 'prompt_rejected'])

In [ ]:
dataset.column_names

In [ ]:
split_dataset = dataset['train'].train_test_split(test_size=0.2)
# Split into train and test splits
dataset_dict = DatasetDict({
    'train': split_dataset['train'],
    'test': split_dataset['test'],
})

### LoRA configuration

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["attn.c_attn", "attn.c_proj"]  #attention layers
)

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=3,
    num_train_epochs=3,
    gradient_accumulation_steps=8,
    learning_rate=1.41e-5,
    output_dir="./model_output3",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
)

### RewardTrainer

In [ ]:
trainer = RewardTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=dataset_dict['train'],
    eval_dataset=dataset_dict['test'],
    peft_config=peft_config,
)

### Model Training

In [ ]:
output_dir="./model_output3"
trainer.train()

In [ ]:
trainer.save_model(output_dir)

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
model.config.save_pretrained("./backup")

### Pre-trained Model

In [ ]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/VZcK8FJ-kQ3nEJoxWGNYTQ/RetriverTrainerModel.zip

In [ ]:
!unzip -o RetriverTrainerModel.zip -d extracted_model


### Model Evaluation

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = GPT2ForSequenceClassification.from_pretrained("./extracted_model/model_output3", num_labels=1).to(DEVICE)
model

In [ ]:
log_file = f"extracted_model/model_output3/checkpoint-2500/trainer_state.json"
with open(log_file, 'r') as f:
    logs = json.load(f)

steps = []
losses = []
for log in logs["log_history"]:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, label="Training Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training Loss Over Time")
plt.legend()
plt.show()

In [ ]:
# Chosen Test Sample
text1=train_str['chosen'][0]
print(text1)

In [ ]:
inputs = tokenizer(text1, return_tensors="pt", padding=True, truncation=True, max_length=512)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.no_grad():
    outputs = model(**inputs)
logit_1 = outputs.logits
print("Score :",logit_1 )

Do the same for the rejected sample


In [ ]:
# Rejected Test Sample
text2=train_str['rejected'][0]
print(text2)

In [ ]:
inputs = tokenizer(text2, return_tensors="pt", padding=True, truncation=True, max_length=512)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.no_grad():
    outputs = model(**inputs)
logit_2 = outputs.logits
print("Score :",logit_2 )

In [ ]:
if logit_1 > logit_2:
    print("--------selected---------")
    print(text1, logit_1.detach().item())
    print("--------rejected---------")
    print(text2, logit_2.detach().item())
else:
    print("selected ")
    print(text2, logit_2.detach().item())
    print("rejected")
    print(text2, logit_2.detach().item())

In [ ]:
def predict_and_get_logits(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits.squeeze().item()  # Assuming binary classification and batch size of 1

    return logits

In [ ]:
# Function to compare two texts
def compare_texts(text1, text2):
    logit1 = predict_and_get_logits(text1)
    logit2 = predict_and_get_logits(text2)

    if logit1 > logit2:
        print("selected---------")
        print(text1, f"score: {logit1}")

        return text1
    else:
        print("selected---------")
        print(text2,  f"score: {logit2}")

        return text2

In [ ]:
# Define the number of samples to evaluate
N = 10

# Initialize a counter for correct selections
correct_selections = 0

# Iterate over the first N pairs of chosen and rejected responses
for chosen, rejected in zip(train_str['chosen'][0:N], train_str['rejected'][0:N]):
    # Print the chosen response for reference
    print("Chosen Response:\n", chosen)

    # Use the compare_texts function to determine which response is better
    selected_text = compare_texts(chosen, rejected)

    # Check if the selected text is the chosen response
    if selected_text == chosen:
        correct_selections += 1

# Calculate the accuracy as the ratio of correct selections to the total number of samples
accuracy = correct_selections / N

# Print the accuracy
print("Accuracy:", accuracy)